# تکلیف ۲-۲ — فاین‌تیون ParsBERT برای دسته‌بندی خبر فارسی

این نوتبوک کل مسیر اجرا را از ابتدا تا انتها در خود دارد: بارگذاری داده، توکنایزر، ساخت مدل، آموزش، ارزیابی روی دادهٔ پنهان، مدل پایهٔ TF-IDF و آزمایش فریزکردن بدنه.

**مدل پایه:** `HooshvareLab/bert-fa-zwnj-base`  •  **دیتاست:** `persian-news-6` (PN-Summary، شش دسته)

> نوتبوک اختیاری است و نمرهٔ مستقیم ندارد؛ برای بررسی دستیاران آموزشی نگه داشته می‌شود. اعداد این نوتبوک همان اعداد اجرای واقعی است که در `REPORT.md` هم آمده‌اند.

## نتیجهٔ اجرای مرجع (روی CPU، بدون GPU)

| سنجه | مقدار |
|---|---|
| پارامترهای مدل | ۱۱۸٬۳۰۱٬۹۵۸ (سر طبقه‌بندی: ۴٬۶۱۴) |
| loss اعتبارسنجی پیش از آموزش | ۱٫۸۰۵۸ (≈ `ln 6`) |
| دقت اعتبارسنجی | **۰٫۸۸۳۳** |
| Macro-F1 اعتبارسنجی | **۰٫۸۸۲۳** |
| Macro-F1 مدل پایهٔ TF-IDF | ۰٫۸۴۸۱ |
| زمان آموزش (۳ دوره) | حدود ۲۷ دقیقه |

## فهرست

۱. راه‌اندازی و تنظیمات
۲. بارگذاری و بررسی داده
۳. توکنایزر (سؤالات ۱٫۱ تا ۱٫۵)
۴. ساخت مدل و شمارش پارامتر (۲٫۱ و ۲٫۲)
۵. آماده‌سازی داده برای آموزش
۶. loss پیش از آموزش (۲٫۳ و ۲٫۴)
۷. تنظیمات و حلقهٔ آموزش (۲٫۵ و ۲٫۶)
۸. نمودار یادگیری (۲٫۷) و تحلیل نرخ یادگیری (۲٫۸)
۹. ارزیابی، ماتریس درهم‌ریختگی و پیش‌بینی آزمون (۳٫۱ و ۳٫۲)
۱۰. مدل پایهٔ TF-IDF (۳٫۳)
۱۱. محک زدن مدل (۴٫۱ تا ۴٫۳)
۱۲. جمع‌بندی و فایل‌های خروجی

> **دربارهٔ خروجی سلول‌ها:** این نوتبوک با خروجی‌های خالی ذخیره شده است (دوباره اجرا نشده)؛ اما جدول‌ها و اعداد هر بخش از اجرای واقعیِ مرجع گرفته شده و در همان‌جا نوشته شده‌اند. با اجرای متوالی سلول‌ها همان اعداد بازتولید می‌شوند (روی CPU حدود ۳۰ دقیقه).

## ۱. راه‌اندازی و تنظیمات

اگر در Colab اجرا می‌کنید، سلول نصب زیر را از حالت توضیح خارج کنید. نسخه‌ها **دقیقاً** همان‌هایی هستند که صورت سؤال خواسته است؛ خروجی توکنایزر به همین نسخه‌ها بستگی دارد.

In [ ]:
# در Colab این خط را از حالت توضیح خارج کنید (همان نسخه‌های خواسته‌شده در صورت سؤال)
# !pip install -q transformers==4.57.1 tokenizers==0.22.1

### مسیرها و محیط اجرا

اگر مدل به‌صورت محلی موجود باشد از آن استفاده می‌شود (حالت آفلاین)، وگرنه از مخزن HuggingFace دانلود می‌شود — بنابراین نوتبوک هم روی همین سیستم و هم در Colab کار می‌کند.

In [ ]:
import os, sys, json, random, time, uuid
from pathlib import Path

# ---- پیدا کردن ریشهٔ پروژه ----
ROOT = Path.cwd()
for cand in [ROOT, ROOT.parent, Path("/content")]:
    if (cand / "persian-news-6").exists() or (cand / "model" / "bert-fa-zwnj-base").exists():
        ROOT = cand
        break

# ---- اگر بسته‌ها به‌صورت محلی نصب شده‌اند، از همان‌ها استفاده کن ----
PKGS = ROOT / ".pkgs"
if PKGS.exists():
    sys.path.insert(0, str(PKGS))

MODEL_DIR = ROOT / "model" / "bert-fa-zwnj-base"
LOCAL_MODEL = MODEL_DIR.exists()
if LOCAL_MODEL:
    os.environ.setdefault("HF_HUB_OFFLINE", "1")
    os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

MODEL_NAME = str(MODEL_DIR) if LOCAL_MODEL else "HooshvareLab/bert-fa-zwnj-base"
DATA_DIR = ROOT / "persian-news-6"
OUT_DIR = ROOT / "run"
OUT_DIR.mkdir(exist_ok=True)

# ---- اگر فقط فایل zip دیتاست موجود باشد، همان‌جا استخراج می‌شود ----
ZIP_PATH = ROOT / "persian-news-6.zip"
if not DATA_DIR.exists() and ZIP_PATH.exists():
    import zipfile
    zipfile.ZipFile(ZIP_PATH).extractall(ROOT)
    print("دیتاست از zip استخراج شد.")

print("ROOT      :", ROOT)
print("MODEL_NAME:", MODEL_NAME)
print("DATA_DIR  :", DATA_DIR, "| exists:", DATA_DIR.exists())

In [ ]:
import numpy as np
import pandas as pd
import torch
import transformers
import tokenizers
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from sklearn.metrics import (f1_score, accuracy_score,
                             classification_report, confusion_matrix)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("transformers:", transformers.__version__)
print("tokenizers  :", tokenizers.__version__)
print("torch       :", torch.__version__)
print("cuda        :", torch.cuda.is_available())

### تنظیمات اجرا

همهٔ ابرپارامترها با مقادیر اجرای مرجع یکی هستند تا نتیجه بازتولید شود (`seed=42`).

In [ ]:
SEED         = 42
MAX_LEN      = 128      # پاسخ سؤال ۱٫۵
BATCH_SIZE   = 16
EPOCHS       = 3
LR           = 3e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
LABEL_ALL    = True

torch.set_num_threads(min(16, os.cpu_count() or 1))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("device:", DEVICE, "| threads:", torch.get_num_threads())

## ۲. بارگذاری و بررسی داده

سه فایل: `train.csv` (۳۰۰۰ سطر با برچسب)، `val.csv` (۶۰۰ سطر با برچسب) و `test.csv` (۱۲۰۰ سطر بدون برچسب). داده‌ها کاملاً متوازن‌اند: هر دسته ۵۰۰ سطر آموزش و ۱۰۰ سطر اعتبارسنجی.

In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

print("shapes:", train_df.shape, val_df.shape, test_df.shape)
print("train columns:", list(train_df.columns), "| test columns:", list(test_df.columns))
print()
print("توزیع برچسب در train:")
print(train_df["label"].value_counts())
print()
print("توزیع برچسب در val:")
print(val_df["label"].value_counts())
print()
print("آمار طول متن (کاراکتر):")
print(train_df["text"].str.len().describe())
print()
print("مقدار گمشده:", train_df.isna().sum().to_dict(), val_df.isna().sum().to_dict(), test_df.isna().sum().to_dict())
print("متن تکراری در train:", int(train_df["text"].duplicated().sum()))
print("متن‌های دارای نیم‌فاصله:", int(train_df["text"].str.contains("\u200c").sum()))

## ۳. توکنایزر (سؤالات ۱٫۱ تا ۱٫۵)

توکنایزر با `AutoTokenizer` بارگذاری می‌شود. با نسخه‌های خواسته‌شده این توکنایزر از نوع `BertTokenizerFast` است (`use_fast=True`).

نکتهٔ کلیدی: ParsBERT نیم‌فاصله را حذف نمی‌کند؛ آن را به یک **توکن مستقل `[ZWNJ]`** تبدیل می‌کند (شناسهٔ ۹ در `tokenizer_config.json`).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("class:", type(tokenizer).__name__, "| is_fast:", tokenizer.is_fast)
print("vocab size:", tokenizer.vocab_size)

### ۱٫۱ — طول توکنی جملهٔ نمونه

پاسخ مرجع: **۲۰ توکن** (یک `[CLS]` + ۱۸ توکن متن + یک `[SEP]`).

In [ ]:
SENT = "مدل\u200cهای زبانی بزرگ، متن\u200cهای فارسی را بهتر از گذشته می\u200cفهمند."

enc = tokenizer(SENT)
toks = tokenizer.convert_ids_to_tokens(enc["input_ids"])
print("جمله:", SENT)
print("تعداد توکن‌ها (با [CLS] و [SEP]):", len(enc["input_ids"]))
print("توکن‌ها:", toks)

### ۱٫۲ — توکن‌های «دانش‌آموزان» (با نیم‌فاصله)

پاسخ مرجع: **گزینهٔ پنجم** — `['[CLS]', 'دانش', '[ZWNJ]', 'آ', '##مو', '##زان', '[SEP]']` (۷ توکن).

توکنایزر اول روی نیم‌فاصله تقسیم می‌کند و بعد `آموزان` را با WordPiece به `آ` + `##مو` + `##زان` می‌شکند.

In [ ]:
WORD = "دانش\u200cآموزان"
w_ids = tokenizer(WORD)["input_ids"]
print("کلمه:", WORD)
print("تعداد توکن:", len(w_ids))
print("خروجی convert_ids_to_tokens:", tokenizer.convert_ids_to_tokens(w_ids))

### ۱٫۳ — رفتار نیم‌فاصلهٔ «مدل‌های»

پاسخ مرجع: **گزینهٔ دوم** — نیم‌فاصله یک توکن جداگانه (`[ZWNJ]`) می‌شود.

سه حالت زیر این را نشان می‌دهد: با نیم‌فاصله، بدون نیم‌فاصله (چسبیده) و با فاصلهٔ کامل.

In [ ]:
for variant in ["مدل\u200cهای", "مدلهای", "مدل های"]:
    ids = tokenizer(variant)["input_ids"]
    print(repr(variant), "->", tokenizer.convert_ids_to_tokens(ids))

### ۱٫۴ و ۱٫۵ — توزیع طول توکن‌ها و انتخاب `max_length`

پاسخ‌های مرجع: بیشترین طول در `train.csv` برابر **۱۷۹** توکن است؛ میانه ۳۳، صدک ۹۵ برابر ۵۹ و صدک ۹۹ برابر ۷۳.

با انتخاب `max_length = 128` فقط **۱ سطر از ۳۰۰۰ (۰٫۰۳٪)** بریده می‌شود و **۹۹٫۹۵٪ کل توکن‌ها** حفظ می‌شوند؛ به همین دلیل ۱۲۸ انتخاب شد: تقریباً هیچ اطلاعاتی از دست نمی‌رود ولی هزینهٔ محاسبه در مقایسه با ۲۵۶ یا ۵۱۲ چند برابر کمتر است.

In [ ]:
lengths = np.array([len(x) for x in tokenizer(train_df["text"].tolist(), add_special_tokens=True,
                                             truncation=False, padding=False)["input_ids"]])
val_lengths  = np.array([len(x) for x in tokenizer(val_df["text"].tolist(), add_special_tokens=True,
                                                  truncation=False, padding=False)["input_ids"]])
test_lengths = np.array([len(x) for x in tokenizer(test_df["text"].tolist(), add_special_tokens=True,
                                                   truncation=False, padding=False)["input_ids"]])

print("بیشترین طول train :", lengths.max())
print("بیشترین طول val   :", val_lengths.max())
print("بیشترین طول test  :", test_lengths.max())
print()
print("میانگین %.1f | میانه %d | صدک۹۰ %d | صدک۹۵ %d | صدک۹۹ %d" % (
    lengths.mean(), np.median(lengths),
    np.percentile(lengths, 90), np.percentile(lengths, 95), np.percentile(lengths, 99)))
print()
print("max_len | درصد متن‌های بریده‌شده | درصد توکن‌های حفظ‌شده")
for ml in [64, 96, 128, 160, 192, 256]:
    truncated = (lengths > ml).mean() * 100
    kept = np.minimum(lengths, ml).sum() / lengths.sum() * 100
    print("%7d | %22.2f%% | %21.2f%%" % (ml, truncated, kept))

plt.figure(figsize=(8, 4), dpi=120)
plt.hist(lengths, bins=60, color="#08519c", alpha=0.85)
plt.axvline(MAX_LEN, color="#cb181d", ls="--", label=f"max_length = {MAX_LEN}")
plt.xlabel("Token length (with specials)"); plt.ylabel("Number of texts")
plt.title("Token-length distribution of train.csv"); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## ۴. ساخت مدل و شمارش پارامتر (۲٫۱ و ۲٫۲)

مدل با `AutoModelForSequenceClassification` و شش برچسب ساخته می‌شود. چک‌پوینت ParsBERT از نوع `BertForMaskedLM` است، پس وزن‌های سر جدید (`classifier`) و همچنین `pooler` به‌صورت تصادفی مقداردهی می‌شوند — این هشدار طبیعی است.

پاسخ‌های مرجع: کل پارامترها **۱۱۸٬۳۰۱٬۹۵۸** و پارامترهای سر طبقه‌بندی **۴٬۶۱۴** (= ۶×۷۶۸ + ۶).

In [ ]:
labels = sorted(train_df["label"].unique().tolist())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
print("برچسب‌ها:", labels)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(labels), id2label=id2label, label2id=label2id
).to(DEVICE)

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_cls   = sum(p.numel() for p in model.classifier.parameters())
print(f"\nکل پارامترها      : {n_total:,}")
print(f"آموزش‌پذیر        : {n_train:,}")
print(f"پارامترهای classifier: {n_cls:,}   (6*768 + 6 = {6*768+6})")
print(f"بدنهٔ BERT        : {n_total - n_cls:,}")

## ۵. آماده‌سازی داده برای آموزش

پدینگ **پویا** است: هر دسته فقط تا بلندترین متن همان دسته پد می‌شود (نه تا `max_length`). این کار روی CPU سرعت آموزش را چند برابر می‌کند، چون میانهٔ طول متن‌ها فقط ۳۳ توکن است.

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = None if labels is None else list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        item = {"text": self.texts[i]}
        if self.labels is not None:
            item["label"] = self.labels[i]
        return item


def collate_fn(batch):
    enc = tokenizer([b["text"] for b in batch], padding="longest", truncation=True,
                    max_length=MAX_LEN, return_tensors="pt")
    if "label" in batch[0]:
        enc["labels"] = torch.tensor([b["label"] for b in batch], dtype=torch.long)
    return enc


train_ds = NewsDataset(train_df["text"], [label2id[l] for l in train_df["label"]])
val_ds   = NewsDataset(val_df["text"],   [label2id[l] for l in val_df["label"]])
test_ds  = NewsDataset(test_df["text"])

g = torch.Generator(); g.manual_seed(SEED)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, generator=g)
val_dl   = DataLoader(val_ds,   batch_size=64,         shuffle=False, collate_fn=collate_fn)
test_dl  = DataLoader(test_ds,  batch_size=64,         shuffle=False, collate_fn=collate_fn)

print("تعداد دسته در train:", len(train_dl), "| در val:", len(val_dl), "| در test:", len(test_dl))

@torch.no_grad()
def evaluate(dl, with_labels=True):
    model.eval()
    total, n = 0.0, 0
    preds, trues, probs = [], [], []
    for batch in dl:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(**batch)
        if with_labels:
            bs = batch["labels"].shape[0]
            total += out.loss.item() * bs
            n += bs
            preds.extend(out.logits.argmax(-1).tolist())
            trues.extend(batch["labels"].tolist())
        probs.append(torch.softmax(out.logits, dim=-1).cpu())
    model.train()
    res = {"loss": total / max(n, 1) if with_labels else None,
           "probs": torch.cat(probs)}
    if with_labels:
        res.update(preds=preds, trues=trues,
                   acc=accuracy_score(trues, preds),
                   macro_f1=f1_score(trues, preds, average="macro"))
    return res

## ۶. loss پیش از آموزش (۲٫۳ و ۲٫۴)

مدل، پیش از هر گام آموزش، روی کل `val.csv` ارزیابی می‌شود.

پاسخ مرجع: **۱٫۸۰۵۸**.

**چرا تقریباً همین عدد؟** سر طبقه‌بندی تصادفی است، پس logits تقریباً صفرند و softmax روی بردار صفر توزیع یکنوای `1/6` می‌دهد؛ در نتیجه Cross-Entropy برابر `-ln(1/6) = ln 6 ≈ 1.7918` می‌شود. عدد اندازه‌گیری‌شده کمی بیشتر است چون logits دقیقاً صفر نیستند.

In [ ]:
pre = evaluate(val_dl)
print(f"loss اعتبارسنجی پیش از آموزش: {pre['loss']:.4f}")
print(f"دقت: {pre['acc']:.4f} | Macro-F1: {pre['macro_f1']:.4f}")
print(f"ln(6) = {np.log(6):.4f}  ← حد نظری برای شش کلاس با logits صفر")

## ۷. تنظیمات و حلقهٔ آموزش (۲٫۵ و ۲٫۶)

بهینه‌ساز `AdamW` با نرخ یادگیری `3e-5`، گرم‌کردن ۱۰٪ گام‌ها و کاهش خطی. وزن‌های bias و LayerNorm از weight decay مستثنی می‌شوند (رویهٔ استاندارد فاین‌تیون BERT).

با ۳۰۰۰ نمونه و دستهٔ ۱۶تایی: **۱۸۸ گام در هر دوره** و **۵۶۴ گام در کل** (warmup = ۵۶ گام).

In [ ]:
steps_per_epoch = len(train_dl)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)

no_decay = ["bias", "LayerNorm.weight"]
optimizer = torch.optim.AdamW(
    [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
         "weight_decay": 0.0},
    ],
    lr=LR, betas=(0.9, 0.999), eps=1e-8,
)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print(f"steps/epoch = {steps_per_epoch} | total_steps = {total_steps} | warmup = {warmup_steps}")
print(f"lr={LR} | batch={BATCH_SIZE} | epochs={EPOCHS} | wd={WEIGHT_DECAY} | max_len={MAX_LEN}")

### حلقهٔ آموزش

loss آموزش هر گام و loss/Macro-F1 اعتبارسنجی در پایان هر دوره ثبت می‌شود. گرادیان‌ها با نُرم ۱٫۰ بریده می‌شوند (`clip_grad_norm_`).

> زمان اجرای مرجع روی CPU: حدود ۵۰۰ ثانیه برای هر دوره، یعنی حدود ۲۷ دقیقه برای سه دوره.

In [ ]:
history = {"steps": [], "epochs": [], "pre_train_val_loss": pre["loss"]}
global_step = 0
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, seen = 0.0, 0
    t_epoch = time.time()

    for batch in train_dl:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(**batch)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        global_step += 1
        running_loss += out.loss.item() * batch["labels"].shape[0]
        seen += batch["labels"].shape[0]
        history["steps"].append({"step": global_step, "epoch": epoch,
                                 "loss": out.loss.item(), "lr": scheduler.get_last_lr()[0]})

        if global_step % 40 == 0:
            print(f"  ep{epoch} step {global_step}/{total_steps} "
                  f"loss={running_loss/seen:.4f} lr={scheduler.get_last_lr()[0]:.2e} "
                  f"elapsed={time.time()-t_start:.0f}s")

    ev = evaluate(val_dl)
    rec = {"epoch": epoch, "train_loss": running_loss / seen,
           "val_loss": ev["loss"], "val_acc": ev["acc"],
           "val_macro_f1": ev["macro_f1"], "time_s": time.time() - t_epoch}
    history["epochs"].append(rec)
    print(f"EPOCH {epoch}: train_loss={rec['train_loss']:.4f} "
          f"val_loss={rec['val_loss']:.4f} val_acc={rec['val_acc']:.4f} "
          f"val_macro_f1={rec['val_macro_f1']:.4f} ({rec['time_s']:.0f}s)")

### جدول دوره‌ها (۲٫۶) — نتیجهٔ اجرای مرجع

| دوره | loss آموزش | loss اعتبارسنجی | Macro-F1 اعتبارسنجی |
|---|---|---|---|
| ۱ | ۰٫۸۲۵۲ | ۰٫۳۶۴۷ | ۰٫۸۷۵۷ |
| ۲ | ۰٫۲۷۹۵ | **۰٫۳۲۷۷** | ۰٫۸۸۲۳ |
| ۳ | **۰٫۱۳۹۶** | ۰٫۳۳۷۲ | **۰٫۸۸۲۳** |

تا دورهٔ دوم هم loss آموزش و هم loss اعتبارسنجی پایین می‌آید؛ از دورهٔ دوم به سوم loss آموزش نصف می‌شود ولی loss اعتبارسنجی بالا می‌رود و F1 بهبودی ندارد — یعنی مدل از این نقطه به بعد train را حفظ می‌کند (شروع overfitting).

## ۸. نمودار یادگیری (۲٫۷) و تحلیل نرخ یادگیری (۲٫۸)

نمودار، loss آموزش (هر گام + میانگین متحرک) و loss اعتبارسنجی (پایان هر دوره) را روی یک شکل با محورهای برچسب‌دار نشان می‌دهد و در `learning_curve.png` ذخیره می‌شود.

In [ ]:
xs = np.array([s["step"] for s in history["steps"]])
ys = np.array([s["loss"] for s in history["steps"]])
w = 10
if len(ys) >= w:                      # میانگین متحرک (برای اجراهای کوتاه هم امن است)
    smooth = np.convolve(ys, np.ones(w) / w, mode="valid")
    sx = xs[w - 1:]
else:
    smooth, sx = ys, xs

fig, ax = plt.subplots(figsize=(9.5, 5.4), dpi=130)
ax.plot(xs, ys, color="#9ecae1", lw=0.9, alpha=0.75, label="Training loss (per step)")
ax.plot(sx, smooth, color="#08519c", lw=2.2, label=f"Training loss (moving avg, w={w})")

ep_x = [e["epoch"] * steps_per_epoch for e in history["epochs"]]
ep_y = [e["val_loss"] for e in history["epochs"]]
ax.plot(ep_x, ep_y, color="#cb181d", marker="o", ms=7, lw=2.2, label="Validation loss (per epoch)")
for x, y in zip(ep_x, ep_y):
    ax.annotate(f"{y:.4f}", (x, y), textcoords="offset points", xytext=(0, 9),
                ha="center", fontsize=9, color="#cb181d", fontweight="bold")
for x in ep_x:
    ax.axvline(x, color="grey", ls=":", lw=0.9, alpha=0.6)

ax.set_xlabel("Optimization step"); ax.set_ylabel("Cross-entropy loss")
ax.set_title("ParsBERT fine-tuning: training vs validation loss", fontweight="bold")
ax.legend(); ax.grid(alpha=0.3, ls="--")
fig.tight_layout()
fig.savefig(OUT_DIR / "learning_curve.png", bbox_inches="tight")
plt.show()

print("Macro-F1 هر دوره:", [round(e["val_macro_f1"], 4) for e in history["epochs"]])

**در نمودار چه می‌بینیم؟** loss آموزش در ۱۵۰ گام اول به‌سرعت از ۱٫۸ به حدود ۰٫۵ سقوط می‌کند و بعد با نوسان زیاد تا حدود ۰٫۱۵ پایین می‌آید. loss اعتبارسنجی در پایان دورهٔ دوم به کف ۰٫۳۲۷۷ می‌رسد و سپس صاف می‌ماند و کمی بالا می‌رود. فاصلهٔ فزایندهٔ این دو منحنی، همان شروع overfitting است.

### ۲٫۸ — چرا `2e-5` تا `5e-5` و چرا ۲ تا ۴ دوره؟

- **نرخ یادگیری کوچک:** وزن‌های ParsBERT از قبل نمایش‌های زبانی خوبی ساخته‌اند؛ فاین‌تیون فقط جابه‌جایی ملایم آن‌ها به‌سمت وظیفهٔ شش‌کلاسی است. نرخ یادگیری بزرگ (مثلاً `1e-3`) باعث فراموشی فاجعه‌بار می‌شود: مدل دانش زبانی‌اش را از دست می‌دهد و روی ۳۰۰۰ نمونه بیش‌برازش می‌کند؛ در عمل انتظار آموزش ناپایدار یا واگرا داریم.
- **چند دورهٔ کم:** با ۳۰۰۰ نمونه، بیشترین جهش در دورهٔ اول رخ می‌دهد و دورهٔ دوم به نقطهٔ تعادل می‌رسد. با ۲۰ دوره، loss آموزش به صفر نزدیک می‌شود ولی loss اعتبارسنجی بالا می‌رود و Macro-F1 افت می‌کند؛ شاهدش همین اجرا است که با فقط یک دورهٔ اضافه، loss اعتبارسنجی از ۰٫۳۲۸ به ۰٫۳۳۷ رفت در حالی که loss آموزش نصف شد. بهترین کاربرد `val.csv` همان early stopping روی دورهٔ دوم است.

## ۹. ارزیابی روی اعتبارسنجی و پیش‌بینی آزمون (۳٫۱ و ۳٫۲)

نتیجهٔ اجرای مرجع: دقت **۰٫۸۸۳۳** و Macro-F1 **۰٫۸۸۲۳**.

پیش‌بینی‌ها روی `test.csv` در `run/predictions.csv` با دو ستون `id,label` ذخیره می‌شوند — همان فایلی که باید در بخش پیوست‌ها بارگذاری شود.

In [ ]:
final = evaluate(val_dl)
print(classification_report(final["trues"], final["preds"], target_names=labels, digits=4))

cm = confusion_matrix(final["trues"], final["preds"])
print("ماتریس درهم‌ریختگی (سطر=واقعی، ستون=پیش‌بینی):")
print(pd.DataFrame(cm, index=labels, columns=labels))
print(f"\nدقت: {final['acc']:.4f} | Macro-F1: {final['macro_f1']:.4f}")

### زوج‌های پراشتباه

بیشترین تداخل بین **«سیاست» و «بین‌الملل»** است (۱۶ اشتباه دوسویه)، پس از آن **اقتصاد↔علم و فناوری**، **سلامت↔علم و فناوری** و **اقتصاد↔سیاست** (هر کدام حدود ۱۰).

دلیل: خبرهای سیاسی و بین‌المللی هر دو دربارهٔ «مقام/سفر/بیانیه» هستند و خلاصهٔ یک‌جمله‌ای دامنهٔ داخلی یا خارجی را صریح نمی‌گوید؛ خبرهای دانش‌بنیان و کسب‌وکار اینترنتی هم‌زمان واژگان فناوری و اقتصاد دارند؛ و خبرهای پزشکی/کرونا هم «علمی» و هم «سلامتی» هستند.

In [ ]:
test_out = evaluate(test_dl, with_labels=False)
test_pred = test_out["probs"].argmax(-1).tolist()

pred_df = pd.DataFrame({"id": test_df["id"], "label": [id2label[i] for i in test_pred]})
pred_df.to_csv(OUT_DIR / "predictions.csv", index=False, encoding="utf-8")

print("predictions.csv ساخته شد | shape:", pred_df.shape)
print(pred_df.head())
print()
print("توزیع پیش‌بینی‌ها:")
print(pred_df["label"].value_counts())

## ۱۰. مدل پایهٔ TF-IDF (۳٫۳)

`TfidfVectorizer` + `LogisticRegression` روی همان `train.csv`. چند تنظیم آزمایش می‌شود و بهترین روی `val.csv` گزارش می‌شود.

نتیجهٔ اجرای مرجع: بهترین حالت `ngram=(1,2)` و `C=3` با Macro-F1 = **۰٫۸۴۸۱** — یعنی فاین‌تیون حدود **۳٫۴ واحد درصد** بهتر است.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

results = []
best = None
for ngram in [(1, 1), (1, 2)]:
    for C in [1.0, 3.0, 10.0]:
        pipe = Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=ngram, min_df=2, sublinear_tf=True,
                                      max_features=200000)),
            ("clf", LogisticRegression(C=C, max_iter=2000)),
        ])
        pipe.fit(train_df["text"], train_df["label"])
        p = pipe.predict(val_df["text"])
        f1 = f1_score(val_df["label"], p, average="macro")
        acc = accuracy_score(val_df["label"], p)
        results.append({"ngram": str(ngram), "C": C, "acc": round(acc, 4), "macro_f1": round(f1, 4)})
        print(f"ngram={ngram} C={C:5.1f}  acc={acc:.4f}  macro-F1={f1:.4f}")
        if best is None or f1 > best["macro_f1"]:
            best = {"ngram": str(ngram), "C": C, "acc": acc, "macro_f1": f1, "pipe": pipe}

print()
print("بهترین مدل پایه:", {k: (round(v, 4) if isinstance(v, float) else v)
                            for k, v in best.items() if k != "pipe"})
print(f"اختلاف با فاین‌تیون: {final['macro_f1'] - best['macro_f1']:.4f}")

**تفسیر اختلاف:** TF-IDF فقط هم‌پوشانی واژگانی را می‌بیند و نحو/نفی/ترتیب را نمی‌فهمد؛ ParsBERT معنای جمله را به‌صورت زمینه‌ای کد می‌کند. در عوض، مدل پایه روی این دیتاست قوی است چون متن‌ها **خلاصهٔ کوتاه خبر** هستند (میانه ۳۳ توکن) و در خلاصهٔ خبر معمولاً واژه‌های کلیدی دسته حاضرند؛ به همین دلیل شکاف ۳٫۴ واحد است و نه ۲۰ واحد.

## ۱۱. محک زدن مدل

### ۴٫۱ — سه خبر مرزی

سه خبر که عمداً بین دو دسته‌اند؛ برای هر کدام برچسب و احتمال softmax را می‌بینیم.

نتیجهٔ اجرای مرجع: «الف» → ورزش (۰٫۹۶۰۳)، «ب» → سلامت (۰٫۹۷۵۴)، «ج» → سیاست (۰٫۹۶۵۲). یعنی مدل روی هر سه **مردد نیست** و با اطمینان بالای ۰٫۹۶ انتخاب می‌کند — که خودش نشانهٔ بیش‌اطمینانی است، چون هر سه جمله از نظر انسانی دوپهلو هستند.

In [ ]:
probes = {
    "الف": "باشگاه استقلال برای جبران زیان مالی، بخشی از سهام خود را در بورس عرضه می‌کند.",
    "ب": "وزیر بهداشت در سفر به ژنو با مدیرکل سازمان جهانی بهداشت دربارهٔ واکسن کرونا گفتگو کرد.",
    "ج": "مجلس طرح حمایت از شرکت‌های دانش‌بنیان در حوزهٔ هوش مصنوعی را تصویب کرد.",
}

model.eval()
with torch.no_grad():
    for key, text in probes.items():
        enc = tokenizer(text, return_tensors="pt", truncation=True,
                        max_length=MAX_LEN).to(DEVICE)
        probs = torch.softmax(model(**enc).logits[0], dim=-1).cpu()
        order = torch.argsort(probs, descending=True)
        print(f"\n({key}) {text}")
        print(f"    پیش‌بینی: {id2label[order[0].item()]}  با احتمال {probs[order[0]]:.4f}")
        print("    سه احتمال اول:", {id2label[i.item()]: round(float(probs[i]), 4) for i in order[:3]})

**تفسیر:** جملهٔ «الف» هم «باشگاه استقلال» (ورزش) دارد و هم «عرضهٔ سهام» (اقتصاد) — مدل به اسم تیم چسبیده (که با برچسب‌دهی این دیتاست هم‌خوان است). جملهٔ «ب» هم «واکسن/وزیر بهداشت» دارد و هم «سازمان جهانی بهداشت در ژنو» (بین‌الملل) و مدل سلامت را گرفته. جملهٔ «ج» هم «مجلس» دارد و هم «هوش مصنوعی» و مدل سیاست را گرفته و احتمال علم و فناوری را فقط ۰٫۰۲۳ گذاشته است.

### ۴٫۲ — سه نمونهٔ اشتباه از `val.csv`

اجرای مرجع: ۷۰ سطر از ۶۰۰ سطر val اشتباه بود. سه نمونهٔ گویا از سه نوع علت متفاوت:

۱. **خطای مدل:** «...آخرین مهلت ارایه اظهارنامه مالیات بر ارزش افزوده...» — واقعی: اقتصاد، پیش‌بینی: سلامت (۰٫۹۰۱۵). هیچ نشانهٔ سلامتی در متن نیست؛ مدل به الگوی «مهلت/دوره/پایان» چسبیده که در خبرهای سلامت این دیتاست پرتکرار است.

۲. **ابهام واقعی خبر:** «همه‌گیری ویروس کرونا فعالیت‌های اقتصادی هنگ کنگ را مختل کرده است.» — واقعی: بین‌الملل، پیش‌بینی: اقتصاد (۰٫۶۴۶۹). موضوع بین‌المللی است ولی گزارهٔ اصلی اقتصادی؛ احتمال کم و نزدیک بودن دو گزینه نشان می‌دهد خود مدل هم مردد بوده.

۳. **برچسب نادرست داده:** «وزیر تعاون، کار و رفاه اجتماعی گفت: مرحله پنجم پرداخت اشتغال پایدار روستایی را آغاز کردیم.» — واقعی: سیاست، پیش‌بینی: اقتصاد (۰٫۹۹۶۱). محتوا کاملاً اقتصادی/رفاهی است و تنها نشانهٔ سیاسی‌اش «وزیر» بودن گوینده است؛ به نظر می‌رسد برچسب بر اساس حضور وزیر زده شده، نه موضوع خبر.

In [ ]:
mis = []
for i, (p, t) in enumerate(zip(final["preds"], final["trues"])):
    if p != t:
        pr = final["probs"][i]
        mis.append({"text": val_df["text"].iloc[i], "true": id2label[t], "pred": id2label[p],
                    "p_pred": round(float(pr[p]), 4), "p_true": round(float(pr[t]), 4)})

print(f"تعداد اشتباه: {len(mis)} از {len(val_df)}")
for m in mis[:5]:
    print(f"\n[{m['true']} -> {m['pred']}]  p_pred={m['p_pred']}  p_true={m['p_true']}")
    print("   ", m["text"][:180])

### ۴٫۳ — فریزکردن بدنهٔ BERT و آموزش فقط سر طبقه‌بندی

همهٔ لایه‌های `model.bert` فریز می‌شوند و فقط ۴٬۶۱۴ پارامتر سر آموزش می‌بیند (با `lr=1e-3`، چون فقط یک لایهٔ خطی است).

نتیجهٔ اجرای مرجع: بعد از یک دوره Macro-F1 = **۰٫۷۵۳۲** در **۱۶۸ ثانیه**، در برابر ۰٫۸۷۵۷ در ۵۱۸ ثانیه برای فاین‌تیون کامل. با ادامه تا سه دوره به سقف **۰٫۷۷۰۵** رسید (۰٫۷۵۳۲ → ۰٫۷۷۰۵ → ۰٫۷۶۵۴) و بعد افت کرد.

In [ ]:
frozen = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(labels), id2label=id2label, label2id=label2id
).to(DEVICE).train()

for p in frozen.bert.parameters():
    p.requires_grad = False
trainable_params = [p for p in frozen.parameters() if p.requires_grad]
print("پارامترهای آموزش‌پذیر پس از فریز:", f"{sum(p.numel() for p in trainable_params):,}")

opt_f = torch.optim.AdamW(trainable_params, lr=1e-3)
FROZEN_EPOCHS = 3
t0 = time.time()

for epoch in range(1, FROZEN_EPOCHS + 1):
    frozen.train()
    running, seen = 0.0, 0
    for batch in train_dl:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = frozen(**batch)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        opt_f.step(); opt_f.zero_grad()
        running += out.loss.item() * batch["labels"].shape[0]
        seen += batch["labels"].shape[0]

    frozen.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in val_dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            preds.extend(frozen(**batch).logits.argmax(-1).tolist())
            trues.extend(batch["labels"].tolist())
    print(f"EPOCH {epoch}: train_loss={running/seen:.4f} "
          f"val_acc={accuracy_score(trues, preds):.4f} "
          f"macro-F1={f1_score(trues, preds, average='macro'):.4f} "
          f"elapsed={time.time()-t0:.0f}s")

**چرا دقت کمتر و زمان کمتر؟** چون بدنه فریز است، گرادیان از آن عبور نمی‌کند و هزینهٔ پس‌انتشار (حدود دو سوم کل محاسبه) حذف می‌شود؛ در نتیجه آموزش حدود ۳ برابر سریع‌تر است. اما نمایش‌های ParsBERT برای «دسته‌بندی موضوعی خبر» بهینه نشده‌اند و یک مرز خطی روی نمایش‌های ثابت، سقف پایینی دارد؛ ظرفیت ۱۱۸ میلیون پارامتری بی‌استفاده می‌ماند.

**نقش `[CLS]`:** این توکن ویژه در ابتدای ورودی می‌آید و چون در لایه‌های توجه به همهٔ توکن‌ها توجه می‌کند، پس از ۱۲ لایه به بردار خلاصهٔ کل جمله تبدیل می‌شود. سر طبقه‌بندی در `BertForSequenceClassification` فقط از همین بردار استفاده می‌کند: `pooler_output = tanh(W·h_[CLS] + b)` و سپس `logits = W_c·pooler_output + b_c`. در فاین‌تیون کامل، گرادیان از همین مسیر به تمام لایه‌ها برمی‌گردد و مدل یاد می‌گیرد `[CLS]` را برای تفکیک شش موضوع بسازد؛ در حالت فریز این امکان وجود ندارد.

## ۱۲. جمع‌بندی و فایل‌های خروجی

| سؤال | پاسخ |
|---|---|
| ۱٫۱ | ۲۰ توکن |
| ۱٫۲ | `['[CLS]', 'دانش', '[ZWNJ]', 'آ', '##مو', '##زان', '[SEP]']` (گزینهٔ پنجم) |
| ۱٫۳ | نیم‌فاصله یک توکن جداگانه می‌شود (گزینهٔ دوم) |
| ۱٫۴ | ۱۷۹ توکن |
| ۱٫۵ | `max_length = 128` (فقط ۱ سطر بریده می‌شود، ۹۹٫۹۵٪ توکن‌ها حفظ می‌شوند) |
| ۲٫۱ | ۱۱۸٬۳۰۱٬۹۵۸ |
| ۲٫۲ | ۴٬۶۱۴ |
| ۲٫۳ | ۱٫۸۰۵۸ |
| ۲٫۴ | `ln 6 ≈ 1.7918` |
| ۲٫۵ | lr=3e-5، batch=16، ۳ دوره، warmup=56، wd=0.01، max_len=128 — ۱۸۸ گام/دوره، ۵۶۴ گام کل |
| ۲٫۶ | جدول بخش ۷ |
| ۲٫۷ | نمودار `learning_curve.png` |
| ۳٫۱ | `run/predictions.csv` (۱۲۰۰ سطر) |
| ۳٫۲ | دقت ۰٫۸۸۳۳ / Macro-F1 ۰٫۸۸۲۳ — پراشتباه‌ترین زوج: سیاست↔بین‌الملل |
| ۳٫۳ | مدل پایه ۰٫۸۴۸۱ (اختلاف ۳٫۴ واحد درصد) |
| ۴٫۱ | ورزش ۰٫۹۶۰۳ / سلامت ۰٫۹۷۵۴ / سیاست ۰٫۹۶۵۲ |
| ۴٫۲ | سه نمونه در بخش ۱۱ |
| ۴٫۳ | فریز: ۰٫۷۷۰۵ و ۳ برابر سریع‌تر |

**خروجی‌ها:**

```
run/predictions.csv      ← فایل ارسالی سؤال ۳٫۱
run/learning_curve.png   ← نمودار سؤال ۲٫۷
REPORT.md                ← گزارش کامل با تحلیل‌ها
```